# Chapitre 12 · Survivre à Colab

Notebook du chapitre 12 de *Construire un LLM de zéro* (Partie III « S'entraîner comme un labo »).

**Comment travailler.** D'abord la leçon : tout le code du chapitre, complet et
exécutable de bout en bout ; lis, exécute, triture (retire une pièce du checkpoint,
change l'intervalle de sauvegarde, regarde ce qui casse). À la fin, la section
**Exercices** : quatre défis à trous, du plus simple au plus costaud, validés par
des `assert`. Les corrigés vivent dans le notebook solution.

Colab coupe les sessions, et tout ce qui vit en mémoire disparaît. Ce chapitre donne
à ton entraînement l'hygiène d'un vrai labo : un **checkpoint** complet, une
**sauvegarde atomique**, une **reprise exacte** prouvée à la décimale près, la
**rotation**, la **journalisation** JSONL et le montage de **Google Drive** sans
casser hors ligne.

Tout tourne sur CPU, hors ligne, en moins d'une minute. Sur Colab, seule la cellule
Google Drive change de comportement : elle monte ton Drive au lieu de retomber sur
un dossier local.

## 0. Le GPT des fables, en petit (fourni)

On reprend le corpus jouet et l'architecture du chapitre 10, retaillés pour quelques
secondes de CPU. Le modèle n'est pas le sujet du chapitre : le sujet, c'est de le
faire **survivre** à une coupure.

In [ ]:
import os
import json
import time
import random
import shutil

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print("PyTorch", torch.__version__)

In [ ]:
# Un extrait de fables, répété pour donner de la matière à apprendre.
# (Le vrai notebook du chapitre 10 embarque les 30 fables complètes.)
fable = """\
LA CIGALE ET LA FOURMI
La cigale, ayant chante
Tout l'ete,
Se trouva fort depourvue
Quand la bise fut venue :
Pas un seul petit morceau
De mouche ou de vermisseau.
Elle alla crier famine
Chez la fourmi, sa voisine,
La priant de lui preter
Quelque grain pour subsister
Jusqu'a la saison nouvelle.
"""
corpus = fable * 40

chars = sorted(set(corpus))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in corpus])
print(f"vocabulaire : {vocab_size} caractères | corpus : {data.shape[0]} tokens")

In [ ]:
block_size = 32
d_model, n_heads, n_layers, d_ff = 64, 4, 2, 256


class BlocTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )

    def forward(self, x, masque):
        a, _ = self.attn(self.ln1(x), self.ln1(x), self.ln1(x),
                         attn_mask=masque, need_weights=False)
        x = x + a                       # résidu après l'attention
        x = x + self.ffn(self.ln2(x))   # résidu après le FFN
        return x


class GPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Embedding(block_size, d_model)
        self.blocs = nn.ModuleList([BlocTransformer() for _ in range(n_layers)])
        self.ln_final = nn.LayerNorm(d_model)
        self.tete = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, x):
        B, T = x.shape
        h = self.tok(x) + self.pos(torch.arange(T))                 # (B, T, d_model)
        masque = torch.triu(torch.full((T, T), float("-inf")), diagonal=1)
        for bloc in self.blocs:
            h = bloc(h, masque)                                     # shape préservée
        return self.tete(self.ln_final(h))                         # (B, T, vocab_size)


n_params = sum(p.numel() for p in GPT().parameters())
print(f"{n_params} paramètres")

## 1. Un checkpoint, c'est plus que les poids

### 1.1 Le réflexe du débutant, et pourquoi il ment

Cherche « save model pytorch » et tu tomberas sur la ligne ci-dessous, mille fois
recopiée. Elle sauve les **weights**, le fichier de nombres appris. Et c'est un piège :
un entraînement, c'est un modèle **plus tout un contexte**. Reprends depuis les seuls
poids et tu repars avec un optimizer neuf (AdamW a oublié ses moments), un scheduler
neuf et un RNG neuf. Rien ne plante : l'entraînement *semble* reprendre, mais sur une
autre trajectoire. On le prouve chiffres à l'appui plus bas.

In [ ]:
# Le réflexe du débutant, tel quel : les poids, et rien d'autre.
modele = GPT()
torch.save(modele.state_dict(), "modele.pt")
print(f"modele.pt : {os.path.getsize('modele.pt')} octets de poids...")
print("mais ni l'optimizer, ni le scheduler, ni les RNG.")
os.remove("modele.pt")   # on efface : ce fichier ne suffit pas à reprendre

### 1.2 Les cinq pièces d'un checkpoint complet

Un checkpoint qui permet une reprise **exacte** embarque cinq choses. Retiens-les
comme une liste de contrôle avant décollage :

1. les poids du modèle : `modele.state_dict()` ;
2. l'état de l'optimizer : `optim.state_dict()`, les moments d'AdamW ;
3. l'état du scheduler : `sched.state_dict()`, la position sur la courbe de learning rate ;
4. le numéro de pas atteint : un simple entier `step` ;
5. l'état des générateurs aléatoires : torch global, notre `Generator`, le module `random`.

### 1.3 Isoler l'aléatoire pour pouvoir le rejouer

Le tirage des batchs consomme de l'aléatoire. Pour une reprise exacte, il faut savoir
*où en est* ce tirage. On ne dépend donc pas du hasard global : on passe à chaque tirage
un `torch.Generator` à nous, dont on peut lire l'état (`get_state`) et le re-poser
(`set_state`). C'est la pièce qui rend la reprise vraiment exacte.

In [ ]:
def fabriquer_batch(gen, taille=16):
    ix = torch.randint(0, len(data) - block_size - 1, (taille,), generator=gen)
    x = torch.stack([data[i : i + block_size] for i in ix])          # (B, T)
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])  # (B, T), décalé de 1
    return x, y

Deux aides pour toute la suite : `construire()` fabrique un modèle, un optimizer et un
scheduler tout neufs à partir de la même graine (deux appels donnent des poids
identiques), et `entrainer()` avance de `n_steps` pas en notant chaque loss dans une liste.

In [ ]:
def construire():
    """Fabrique un modèle, un optimizer et un scheduler tout neufs, toujours
    à partir de la même graine : deux appels donnent des poids identiques."""
    torch.manual_seed(42)
    random.seed(42)
    modele = GPT()
    optim = torch.optim.AdamW(modele.parameters(), lr=3e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=60)
    return modele, optim, sched


def entrainer(modele, optim, sched, gen, n_steps, journal):
    for _ in range(n_steps):
        x, y = fabriquer_batch(gen)
        loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))
        optim.zero_grad(); loss.backward(); optim.step(); sched.step()
        journal.append(round(loss.item(), 4))
    return journal

## 2. Sauver sans se faire couper en pleine écriture

`torch.save` écrit octet par octet. Si Colab coupe **pendant** l'écriture, tu obtiens un
fichier tronqué, illisible, qui a écrasé le bon. La parade des labos : écrire dans un
fichier `.tmp`, puis renommer avec `os.replace`, qui est **atomique** sur un même disque
(il réussit d'un coup ou pas du tout). Ton `checkpoint.pt` est toujours soit l'ancien
complet, soit le nouveau complet, jamais un à-moitié.

In [ ]:
def sauver_checkpoint(chemin, modele, optim, sched, gen, step):
    paquet = {
        "model": modele.state_dict(),
        "optim": optim.state_dict(),
        "sched": sched.state_dict(),
        "step": step,
        "torch_rng": torch.get_rng_state(),   # RNG global de torch
        "gen_rng": gen.get_state(),           # RNG de notre tirage de batchs
        "py_rng": random.getstate(),          # RNG du module random de Python
    }
    tmp = chemin + ".tmp"
    torch.save(paquet, tmp)
    os.replace(tmp, chemin)   # renommage atomique : jamais de fichier à moitié écrit


def charger_checkpoint(chemin, modele, optim, sched, gen):
    paquet = torch.load(chemin)
    modele.load_state_dict(paquet["model"])
    optim.load_state_dict(paquet["optim"])
    sched.load_state_dict(paquet["sched"])
    torch.set_rng_state(paquet["torch_rng"])
    gen.set_state(paquet["gen_rng"])
    random.setstate(paquet["py_rng"])
    return paquet["step"]

## 3. La preuve : une reprise exacte

Le protocole tient en deux courses sur le même modèle. Une **course de référence** :
60 pas d'un seul trait. Une **course coupée** : 30 pas, un checkpoint complet, on
**jette tout** en mémoire, on reconstruit à neuf, on recharge, et on repart pour
30 pas. Si le checkpoint est complet, les 60 loss doivent être **identiques**.
Pas « proches » : identiques.

In [ ]:
# --- référence : 60 pas d'un trait ---
modele, optim, sched = construire()
gen = torch.Generator().manual_seed(123)   # graine du tirage des batchs
loss_reference = entrainer(modele, optim, sched, gen, 60, [])

print("premières loss :", loss_reference[:3])
print("loss au pas 30 :", loss_reference[30])
print("dernière loss  :", loss_reference[-1])

In [ ]:
DOSSIER = "checkpoints_demo"
os.makedirs(DOSSIER, exist_ok=True)

# --- première moitié : 30 pas, puis on sauve tout ---
modele, optim, sched = construire()
gen = torch.Generator().manual_seed(123)
loss_repris = entrainer(modele, optim, sched, gen, 30, [])
sauver_checkpoint(f"{DOSSIER}/etape_30.pt", modele, optim, sched, gen, step=30)
print(f"Checkpoint écrit au pas 30. loss courante : {loss_repris[-1]}")

# --- la coupure : on jette TOUT et on reconstruit à neuf ---
del modele, optim, sched, gen

modele, optim, sched = construire()          # poids neufs
gen = torch.Generator()                      # générateur neuf, état quelconque
step = charger_checkpoint(f"{DOSSIER}/etape_30.pt", modele, optim, sched, gen)
print(f"Reprise au pas {step}.")

# --- seconde moitié : 30 pas de plus ---
loss_repris = entrainer(modele, optim, sched, gen, 30, loss_repris)

In [ ]:
# La preuve : les deux courbes sont-elles identiques ?
ecart_max = max(abs(a - b) for a, b in zip(loss_reference, loss_repris))
print(f"nombre de pas comparés : {len(loss_reference)}")
print(f"écart maximum entre la course d'un trait et la course coupée/reprise : {ecart_max}")
assert loss_reference == loss_repris, "les courbes diffèrent : le checkpoint est incomplet"
print("Identiques au centième près : la reprise est EXACTE.")

Écart maximum : **0.0**. La course coupée et reprise est indistinguable de la course
d'un seul trait. Colab peut te couper au pas 30, tu ne perds rien, pas même une
décimale. C'est ça, l'objectif : que la coupure devienne un non-événement.

## 4. Le cas qui échoue : reprendre sans l'optimizer ni le RNG

La faute classique des tutoriels : sauver **seulement les poids**. À la reprise,
l'optimizer est neuf (AdamW a perdu ses moments) et le tirage des batchs repart d'une
autre graine. On mesure de combien la courbe s'écarte de la reprise exacte.

In [ ]:
# --- même première moitié : 30 pas ---
modele, optim, sched = construire()
gen = torch.Generator().manual_seed(123)
loss_casse = entrainer(modele, optim, sched, gen, 30, [])

# --- le checkpoint PAUVRE : seulement les poids ---
torch.save(modele.state_dict(), f"{DOSSIER}/poids_seuls.pt")
del modele, optim, sched, gen

# --- reprise bancale : poids ok, mais optimizer NEUF et RNG non restauré ---
modele, optim, sched = construire()          # optimizer AdamW neuf : moments à zéro
modele.load_state_dict(torch.load(f"{DOSSIER}/poids_seuls.pt"))
gen = torch.Generator().manual_seed(999)     # mauvaise graine : autres batchs
loss_casse = entrainer(modele, optim, sched, gen, 30, loss_casse)

# --- comparaison avec la reprise exacte ---
print("Reprise EXACTE   , pas 30..34 :", loss_repris[30:35])
print("Reprise BANCALE  , pas 30..34 :", loss_casse[30:35])
derive = round(sum(abs(a - b) for a, b in zip(loss_repris[30:], loss_casse[30:])) / 30, 4)
print(f"écart moyen sur les 30 pas après reprise : {derive}")

La reprise bancale part sur une **autre trajectoire** : l'optimizer retâtonne, les
batchs ne sont plus les mêmes, et l'écart moyen se creuse. Aucune erreur rouge, aucune
exception : c'est le bug qui ne crie pas. Le seul moyen de le détecter, c'est de
comparer à une reprise exacte, comme ici. D'où la règle : sauve les cinq pièces, toujours.

## 5. Ne pas saturer le disque : la rotation

Un checkpoint tous les 500 pas pendant une nuit, et ton Drive sature avant le matin.
La règle des labos : **garder les N derniers seulement**. La subtilité est dans le
`key` du `sorted` : on trie par le **nombre** de pas extrait du nom, pas
alphabétiquement (sinon `etape_100` passerait avant `etape_20`).

In [ ]:
def rotation(dossier, garder=3):
    """Ne conserve que les `garder` checkpoints etape_*.pt les plus récents."""
    fichiers = sorted(
        (f for f in os.listdir(dossier) if f.startswith("etape_") and f.endswith(".pt")),
        key=lambda f: int(f[len("etape_"):-len(".pt")]),
    )
    for vieux in fichiers[:-garder]:
        os.remove(os.path.join(dossier, vieux))
    return sorted(
        (f for f in os.listdir(dossier) if f.startswith("etape_")),
        key=lambda f: int(f[len("etape_"):-len(".pt")]),
    )


# On repart d'un dossier propre pour la démo de rotation.
for f in os.listdir(DOSSIER):
    if f.startswith("etape_"):
        os.remove(os.path.join(DOSSIER, f))

# On simule une longue course : un checkpoint tous les 20 pas.
modele, optim, sched = construire()
gen = torch.Generator().manual_seed(7)
journal = []
for depart in range(0, 100, 20):
    entrainer(modele, optim, sched, gen, 20, journal)
    sauver_checkpoint(f"{DOSSIER}/etape_{depart + 20}.pt", modele, optim, sched, gen, depart + 20)
    restants = rotation(DOSSIER, garder=3)
    print(f"pas {depart + 20:3d} sauvé | checkpoints gardés : {restants}")

La fenêtre glisse : à chaque sauvegarde, le plus vieux checkpoint disparaît, on garde
toujours les trois derniers. Le disque ne gonfle jamais, même sur une course de
plusieurs jours.

## 6. Garder une trace : la journalisation

Colab efface la sortie des cellules à la reconnexion : les `print` ne survivent pas.
Pour garder une trace de la loss, on écrit chaque mesure dans un fichier **JSONL**
(JSON Lines) : une ligne = un objet JSON = une mesure. On ajoute en mode `"a"`
(*append*), on ne réécrit jamais : ce qui est écrit reste écrit, même si la session meurt.

In [ ]:
def journaliser(chemin, **champs):
    """Ajoute une ligne JSON au fichier de log. Une ligne = une mesure."""
    with open(chemin, "a") as f:
        f.write(json.dumps(champs) + "\n")


CHEMIN_LOG = f"{DOSSIER}/journal.jsonl"
open(CHEMIN_LOG, "w").close()   # on repart d'un journal vide pour la démo

modele, optim, sched = construire()
gen = torch.Generator().manual_seed(2024)
debut = time.time()
for step in range(40):
    x, y = fabriquer_batch(gen)
    loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))
    optim.zero_grad(); loss.backward(); optim.step(); sched.step()
    if step % 10 == 0:
        journaliser(
            CHEMIN_LOG,
            step=step,
            loss=round(loss.item(), 4),
            lr=round(sched.get_last_lr()[0], 6),
            secondes=round(time.time() - debut, 2),
        )

print("Contenu du journal :")
print(open(CHEMIN_LOG).read())

Chaque ligne est autonome et lisible à l'œil. On la relit pour tracer la courbe,
comparer deux runs, ou reprendre un tableau. Un CSV ferait aussi l'affaire ; le JSONL
a l'avantage d'accepter des champs qui varient d'une ligne à l'autre sans rien casser.

In [ ]:
# Relire le journal et retrouver la dernière loss enregistrée.
mesures = [json.loads(ligne) for ligne in open(CHEMIN_LOG)]
print(f"{len(mesures)} mesures enregistrées")
print("dernière mesure :", mesures[-1])

**Comme les vrais labos.** Ta journalisation maison fait le travail. Les labos utilisent
des outils dédiés qui font la même chose en plus joli et en ligne : **TensorBoard**
(tableau de bord local) et **Weights & Biases** (`wandb`, le même dans le cloud, avec
partage d'équipe). Le principe est identique à ton JSONL : à chaque pas, tu logges un
dictionnaire de métriques, `wandb.log({"loss": ...})`. Le jour où tu passes à `wandb`,
tu ne remplaces que la fonction `journaliser`.

## 7. Regarder la loss en direct

Tu ne lances pas une course de six heures à l'aveugle : tu veux **voir** la loss
descendre, pour couper tôt si elle explose ou stagne. Le plus simple, sans rien
installer : un `print` à intervalle régulier, avec une **moyenne glissante** qui lisse
le bruit du tirage.

In [ ]:
modele, optim, sched = construire()
gen = torch.Generator().manual_seed(1)
fenetre = []
for step in range(50):
    x, y = fabriquer_batch(gen)
    loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))
    optim.zero_grad(); loss.backward(); optim.step(); sched.step()
    fenetre.append(loss.item())
    if step % 10 == 9:
        moyenne = sum(fenetre[-10:]) / 10          # loss lissée sur 10 pas
        barre = "#" * int((3.5 - moyenne) * 12)    # une jauge qui grandit
        print(f"pas {step + 1:3d} | loss {moyenne:5.3f} | {barre}")

La loss brute saute à chaque batch ; la moyenne sur dix pas laisse voir la vraie
tendance. Sur Colab, tu remplacerais le `print` par une courbe `matplotlib` re-dessinée
à chaque intervalle (avec `IPython.display.clear_output`). L'idée est la même : jamais
de course sans un signal de vie sous les yeux.

**Rappel seeds** : la reprise exacte de ce chapitre repose entièrement sur les graines.
Re-poser l'état des générateurs, c'est faire rejouer aux tirages la même suite exacte.
Sans ça, même avec les poids et l'optimizer, tes batchs divergeraient.

## 8. Monter Google Drive sans casser hors ligne

Sur Colab, le disque de la session **disparaît** avec elle : pour survivre, les
checkpoints doivent aller sur ton Google Drive. Mais `from google.colab import drive`
n'existe que sur Colab. La règle du repli offline de ce livre : une garde
`try / except`, et le même notebook tourne dans les deux mondes, aucune ligne à
commenter à la main.

In [ ]:
def dossier_de_sauvegarde():
    """Sur Colab : monte le Drive et renvoie un dossier dessus.
    Ailleurs (ta machine, ce notebook hors ligne) : un dossier local."""
    try:
        from google.colab import drive          # n'existe que sur Colab
        drive.mount("/content/drive")
        base = "/content/drive/MyDrive/llm_de_zero/checkpoints"
    except Exception:
        base = "checkpoints_local"               # repli hors ligne
    os.makedirs(base, exist_ok=True)
    return base


print("Les checkpoints iront dans :", dossier_de_sauvegarde())

Hors ligne, l'import échoue, le `except` prend le relais, et on continue sur un dossier
local. Sur Colab, le premier `drive.mount` demande une autorisation (une fenêtre, un
clic) : lance cette cellule tôt, avant de commencer une longue course, pour ne pas te
faire interrompre au bout de trois heures.

## 9. La boucle qui survit

On assemble tout. La boucle finale **reprend** depuis le dernier checkpoint s'il
existe, **journalise** chaque mesure, **sauve** un checkpoint complet à intervalle
régulier, et fait la **rotation**. C'est le squelette que tu garderas pour tous les
entraînements de la suite du livre.

In [ ]:
def boucle_qui_survit(dossier, total_steps=60, tous_les=20):
    os.makedirs(dossier, exist_ok=True)
    modele, optim, sched = construire()
    gen = torch.Generator().manual_seed(123)
    log = os.path.join(dossier, "journal.jsonl")

    # --- reprise : si un checkpoint existe, on repart de là ---
    derniers = sorted(
        (f for f in os.listdir(dossier) if f.startswith("etape_") and f.endswith(".pt")),
        key=lambda f: int(f[len("etape_"):-len(".pt")]),
    )
    depart = 0
    if derniers:
        depart = charger_checkpoint(os.path.join(dossier, derniers[-1]),
                                    modele, optim, sched, gen)
        print(f"Reprise depuis {derniers[-1]} (pas {depart}).")
    else:
        open(log, "w").close()
        print("Aucun checkpoint : départ à zéro.")

    for step in range(depart, total_steps):
        x, y = fabriquer_batch(gen)
        loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))
        optim.zero_grad(); loss.backward(); optim.step(); sched.step()
        journaliser(log, step=step, loss=round(loss.item(), 4))
        if (step + 1) % tous_les == 0:
            sauver_checkpoint(os.path.join(dossier, f"etape_{step + 1}.pt"),
                              modele, optim, sched, gen, step + 1)
            rotation(dossier, garder=3)

In [ ]:
# La preuve que la reprise se déclenche : une course jusqu'au pas 40,
# puis on relance la MÊME fonction en visant 60. Elle doit repartir du pas 40.
shutil.rmtree("run_survie", ignore_errors=True)
boucle_qui_survit("run_survie", total_steps=40)
boucle_qui_survit("run_survie", total_steps=60)

Le second appel voit le checkpoint du pas 40 et repart de là, sans qu'on ait rien
changé. C'est le comportement que tu veux sur Colab : tu relances la même cellule
après une coupure, et elle reprend toute seule où elle en était.

In [ ]:
# Nettoyage des dossiers de démonstration, et récap des chiffres clés de la leçon.
for d in ("checkpoints_demo", "checkpoints_local", "run_survie"):
    shutil.rmtree(d, ignore_errors=True)
print("Leçon exécutée jusqu'au bout. Récap :")
print(f"  reprise exacte  : écart maximum = {ecart_max}")
print(f"  reprise bancale : écart moyen après reprise = {derive}")

## Exercices

À toi de jouer : quatre exercices, du plus simple (●) au plus costaud (●●●). La leçon
t'a montré chaque geste ; ici, tu les refais de mémoire. Les exercices redéfinissent
des fonctions de la leçon : c'est voulu, et la cellule de validation (`assert`) qui
suit chaque trou te dit aussitôt si ta version tient debout.

**Le pacte « IA débranchée » agit ici** : pas d'IA pour remplir les trous à ta place.
Elle peut t'expliquer une erreur ; les doigts sur le clavier, c'est toi. Les réponses
sont dans le notebook solution (`solutions/partie_3_sentrainer_comme_un_labo/`),
à n'ouvrir qu'après avoir vraiment essayé.

### Exercice 1 · Journalisation JSONL et rotation — niveau ●

Écris `journaliser` (une ligne JSON = une mesure, ajoutée en mode `"a"`, *append*)
et complète `rotation` (ne garder que les `garder` checkpoints `etape_*.pt` les plus
récents). Deux trous d'une ligne chacun : `json.dumps` d'un côté, `os.remove` de l'autre.

In [ ]:
def journaliser(chemin, **champs):
    with open(chemin, "a") as f:
        ...              # TODO(toi) : écris json.dumps(champs) + un retour à la ligne


def rotation(dossier, garder=3):
    fichiers = sorted(
        (f for f in os.listdir(dossier) if f.startswith("etape_") and f.endswith(".pt")),
        key=lambda f: int(f[len("etape_"):-len(".pt")]),
    )
    for vieux in fichiers[:-garder]:
        ...              # TODO(toi) : supprime le vieux checkpoint (os.remove + os.path.join)
    return sorted(
        (f for f in os.listdir(dossier) if f.startswith("etape_")),
        key=lambda f: int(f[len("etape_"):-len(".pt")]),
    )

In [ ]:
# Validation : journalisation et rotation.
os.makedirs("test_log", exist_ok=True)
open("test_log/j.jsonl", "w").close()
journaliser("test_log/j.jsonl", step=0, loss=3.14)
journaliser("test_log/j.jsonl", step=1, loss=2.71)
lignes = [json.loads(l) for l in open("test_log/j.jsonl")]
assert len(lignes) == 2 and lignes[1]["loss"] == 2.71, "le journal JSONL n'a pas 2 lignes valides"

for s in (20, 40, 60, 80):
    open(f"test_log/etape_{s}.pt", "w").close()
restants = rotation("test_log", garder=3)
assert restants == ["etape_40.pt", "etape_60.pt", "etape_80.pt"], f"rotation KO : {restants}"
shutil.rmtree("test_log", ignore_errors=True)
print("Exercice 1 validé : JSONL et rotation OK")

### Exercice 2 · Le checkpoint pauvre, mesurer la dérive — niveau ●

Refais le cas qui échoue de tes mains : sauve **seulement les poids**, reprends avec un
optimizer neuf et une mauvaise graine de RNG, puis mesure la dérive par rapport à la
reprise exacte (`loss_repris`, la liste des 60 loss). Rien ne plante : c'est ça le piège.

In [ ]:
modele, optim, sched = construire()
gen = torch.Generator().manual_seed(123)
loss_casse = entrainer(modele, optim, sched, gen, 30, [])

os.makedirs("checkpoints_demo", exist_ok=True)
# TODO(toi) : sauve SEULEMENT les poids dans "checkpoints_demo/poids_seuls.pt"
...

del modele, optim, sched, gen
modele, optim, sched = construire()          # optimizer NEUF
# TODO(toi) : recharge seulement les poids dans le modele
...
gen = torch.Generator().manual_seed(999)     # mauvaise graine : autres batchs
loss_casse = entrainer(modele, optim, sched, gen, 30, loss_casse)

In [ ]:
# Validation : la reprise bancale doit s'écarter de la reprise exacte.
derive = round(sum(abs(a - b) for a, b in zip(loss_repris[30:], loss_casse[30:])) / 30, 4)
assert derive > 0.05, (
    f"dérive {derive} trop faible : la reprise bancale devrait s'écarter de la reprise exacte"
)
print(f"Exercice 2 validé : la reprise bancale dérive de {derive} en moyenne "
      "(l'exacte, elle, reste à 0.0)")

### Exercice 3 · Le checkpoint complet et la sauvegarde atomique — niveau ●●

Réécris de mémoire les deux fonctions centrales du chapitre. Les cinq pièces dans le
paquet (poids, optimizer, scheduler, pas, RNG), l'écriture dans un `.tmp` puis le
renommage atomique avec `os.replace`. La validation rejoue le protocole complet
(60 pas d'un trait contre 30 + 30 avec coupure) et exige un écart de 0.0.

In [ ]:
def sauver_checkpoint(chemin, modele, optim, sched, gen, step):
    paquet = {
        "model": ...,        # TODO(toi) : les poids du modèle
        "optim": ...,        # TODO(toi) : l'état de l'optimizer (moments d'AdamW)
        "sched": ...,        # TODO(toi) : l'état du scheduler
        "step": step,
        "torch_rng": torch.get_rng_state(),
        "gen_rng": ...,      # TODO(toi) : l'état de ton Generator (gen.get_state())
        "py_rng": random.getstate(),
    }
    tmp = chemin + ".tmp"
    torch.save(paquet, tmp)
    os.replace(tmp, chemin)  # renommage atomique


def charger_checkpoint(chemin, modele, optim, sched, gen):
    paquet = torch.load(chemin)
    modele.load_state_dict(paquet["model"])
    ...                      # TODO(toi) : recharge optim, sched
    ...
    torch.set_rng_state(paquet["torch_rng"])
    ...                      # TODO(toi) : recharge l'état de gen et de random
    ...
    return paquet["step"]

In [ ]:
# Validation : le protocole complet, référence contre course coupée/reprise.
os.makedirs("checkpoints_demo", exist_ok=True)

# --- référence : 60 pas d'un trait ---
modele, optim, sched = construire()
gen = torch.Generator().manual_seed(123)
loss_reference = entrainer(modele, optim, sched, gen, 60, [])

# --- même course, coupée en 30 + 30 avec checkpoint ---
modele, optim, sched = construire()
gen = torch.Generator().manual_seed(123)
loss_repris = entrainer(modele, optim, sched, gen, 30, [])
sauver_checkpoint("checkpoints_demo/etape_30.pt", modele, optim, sched, gen, step=30)
del modele, optim, sched, gen

modele, optim, sched = construire()
gen = torch.Generator()
step = charger_checkpoint("checkpoints_demo/etape_30.pt", modele, optim, sched, gen)
loss_repris = entrainer(modele, optim, sched, gen, 30, loss_repris)

ecart_max = max(abs(a - b) for a, b in zip(loss_reference, loss_repris))
assert step == 30, f"le checkpoint doit rendre step=30, pas {step}"
assert ecart_max == 0.0, (
    f"écart max {ecart_max} != 0 : ton checkpoint est incomplet "
    "(optimizer, scheduler ou RNG manquant)"
)
print("Exercice 3 validé : reprise EXACTE, écart maximum =", ecart_max)

### Exercice 4 · La boucle qui survit — niveau ●●●

Assemble tout : la boucle **reprend** depuis le dernier checkpoint s'il existe,
**journalise** chaque mesure, **sauve** un checkpoint complet tous les `tous_les` pas
et fait la **rotation** (`garder=3`). Les deux trous sont au cœur de la boucle :
la sauvegarde et la rotation, au bon moment et avec le bon numéro de pas.

In [ ]:
def boucle_qui_survit(dossier, total_steps=60, tous_les=20):
    os.makedirs(dossier, exist_ok=True)
    modele, optim, sched = construire()
    gen = torch.Generator().manual_seed(123)
    log = os.path.join(dossier, "journal.jsonl")

    derniers = sorted(
        (f for f in os.listdir(dossier) if f.startswith("etape_") and f.endswith(".pt")),
        key=lambda f: int(f[len("etape_"):-len(".pt")]),
    )
    depart = 0
    if derniers:
        depart = charger_checkpoint(os.path.join(dossier, derniers[-1]),
                                    modele, optim, sched, gen)
    else:
        open(log, "w").close()

    derniere_loss = None
    for step in range(depart, total_steps):
        x, y = fabriquer_batch(gen)
        loss = F.cross_entropy(modele(x).view(-1, vocab_size), y.view(-1))
        optim.zero_grad(); loss.backward(); optim.step(); sched.step()
        derniere_loss = loss.item()
        journaliser(log, step=step, loss=round(derniere_loss, 4))
        if (step + 1) % tous_les == 0:
            # TODO(toi) : sauve un checkpoint complet, puis fais la rotation (garder=3)
            ...
            ...
    return depart, derniere_loss

In [ ]:
# Validation : la boucle doit reprendre toute seule au bon pas.
shutil.rmtree("run_survie", ignore_errors=True)
depart1, _ = boucle_qui_survit("run_survie", total_steps=40)   # part de 0
depart2, _ = boucle_qui_survit("run_survie", total_steps=60)   # doit reprendre à 40
assert depart1 == 0, f"premier appel : départ {depart1} != 0"
assert depart2 == 40, f"second appel : la reprise devrait partir de 40, pas {depart2}"
print("Exercice 4 validé : la boucle reprend toute seule au bon pas")

# Nettoyage final.
for d in ("checkpoints_demo", "run_survie"):
    shutil.rmtree(d, ignore_errors=True)

## Verdict

Quatre validations vertes : ton entraînement sait se sauver sans se corrompre, se
relire, et repartir exactement d'où il s'était arrêté. La coupure Colab est devenue
un non-événement.

La différence entre quelqu'un qui a lu sur les LLM et quelqu'un qui en a vraiment
entraîné est souvent là. Garde la `boucle_qui_survit` sous la main : tu vas la
réutiliser pour toutes les courses sérieuses de la suite. Au chapitre 13, on s'attaque
au carburant : les données.